# Interactive 2D-McDA viewer

This notebook opens a 2D-McDA NetCDF product, locates the corresponding CALIOP L1 file, loads the four attenuated-backscatter signals at both 333 m × 30 m and 5 km × 60 m resolution, and displays twelve interactive panels with linked profile–altitude axes. Zooming or panning in one panel automatically updates all other panels.

Run the notebook from the `twod-mcda` environment. Copy `visualization/config.example.yaml` to `visualization/config.yaml`, then edit the local configuration without modifying this notebook. `netcdf_path` may remain `null` to select the most recently modified product under `data/output`. `l1_path` may remain `null` when the common processing configuration correctly describes the CALIOP archive.

In [ ]:
from datetime import datetime
from functools import reduce
from pathlib import Path
import re

import cmlidar
import holoviews as hv
import hvplot.xarray  # Register the hvPlot API for xarray objects.
import numpy as np
import xarray as xr
import yaml
import seaborn as sns
from bokeh.models import FixedTicker, HoverTool
from matplotlib.colors import to_hex

from twod_mcda.caliop.constants import CALIOP_L1_PRODUCT_TYPE
from twod_mcda.caliop.discovery import find_granule_file
from twod_mcda.caliop.reader import CALIOPRegularGridReader

hv.extension("bokeh")

In [ ]:
def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
VISUALIZATION_DIRECTORY = PROJECT_ROOT / "visualization"
VIEWER_CONFIG_PATH = VISUALIZATION_DIRECTORY / "config.yaml"
EXAMPLE_CONFIG_PATH = VISUALIZATION_DIRECTORY / "config.example.yaml"

# Fall back to the tracked example so that the notebook works immediately.
active_config_path = (
    VIEWER_CONFIG_PATH if VIEWER_CONFIG_PATH.exists() else EXAMPLE_CONFIG_PATH
)
with active_config_path.open() as stream:
    viewer_config = yaml.safe_load(stream) or {}


def resolve_project_path(value):
    if value is None:
        return None
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (PROJECT_ROOT / path).resolve()


NETCDF_PATH = resolve_project_path(viewer_config.get("netcdf_path"))
L1_PATH = resolve_project_path(viewer_config.get("l1_path"))
COMMON_CONFIG_PATH = resolve_project_path(
    viewer_config.get("common_config_path", "config/common.yaml")
)

plot_config = viewer_config.get("plot", {})
PLOT_WIDTH = int(plot_config.get("width", 500))
PLOT_HEIGHT = int(plot_config.get("height", 280))
configured_altitude_range = plot_config.get("altitude_range")
ALTITUDE_RANGE = (
    tuple(configured_altitude_range)
    if configured_altitude_range is not None
    else None
)

if NETCDF_PATH is None:
    candidates = sorted(
        (PROJECT_ROOT / "data/output").rglob("*.nc"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(
            "No NetCDF was found under data/output; set NETCDF_PATH."
        )
    NETCDF_PATH = candidates[0]
else:
    NETCDF_PATH = Path(NETCDF_PATH).expanduser().resolve()

print(f"Viewer configuration: {active_config_path}")
NETCDF_PATH

In [ ]:
MASK_VARIABLES = (
    "Parallel_Detection_Flags_532",
    "Perpendicular_Detection_Flags_532",
    "Detection_Flags_1064",
    "Composite_Detection_Flags",
)

SIGNAL_VARIABLES = (
    "Total_Attenuated_Backscatter_532",
    "Parallel_Attenuated_Backscatter_532",
    "Perpendicular_Attenuated_Backscatter_532",
    "Attenuated_Backscatter_1064",
)

GRANULE_PATTERN = re.compile(
    r"(\d{4}-\d{2}-\d{2}T\d{2}-\d{2}-\d{2}Z[DN])"
)
L1_FILENAME_PATTERN = re.compile(
    r"CAL_LID_L1-([^-]+)-(V\d+-\d+)\."
    r"(\d{4}-\d{2}-\d{2}T\d{2}-\d{2}-\d{2}Z[DN])\.hdf$"
)


def granule_id_from_product(path, dataset):
    for text in (path.name, str(dataset.attrs.get("id", ""))):
        match = GRANULE_PATTERN.search(text)
        if match is not None:
            return match.group(1)
    raise ValueError(
        "The CALIOP timestamp is absent from the filename and global id attribute."
    )


def locate_l1_file(product_path, dataset, config_path, explicit_path=None):
    if explicit_path is not None:
        path = Path(explicit_path).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    with Path(config_path).open() as stream:
        config = yaml.safe_load(stream)

    granule_id = granule_id_from_product(product_path, dataset)
    granule_time = datetime.strptime(granule_id[:19], "%Y-%m-%dT%H-%M-%S")
    return find_granule_file(config, granule_time)


def l1_file_metadata(path):
    match = L1_FILENAME_PATTERN.fullmatch(path.name)
    if match is None:
        raise ValueError(f"Unrecognized CALIOP L1 filename: {path.name}")
    data_type, encoded_version, granule_id = match.groups()
    version = encoded_version.replace("-", ".", 1)
    return data_type, version, granule_id


def normalize_product_dataset(dataset):
    missing = [name for name in MASK_VARIABLES if name not in dataset]
    if missing:
        raise KeyError(f"Variables missing from the 2D-McDA product: {missing}")

    rename = {}
    if "Profile_ID" in dataset.dims:
        rename["Profile_ID"] = "profile"
    if "Altitude" in dataset.dims:
        rename["Altitude"] = "altitude"
    normalized = dataset.rename(rename)
    return normalized.set_coords(
        [name for name in ("Latitude", "Longitude") if name in normalized]
    )


def load_l1_signals(l1_path, profiles, altitudes, grid):
    data_type, version, granule_id = l1_file_metadata(l1_path)
    profile_start = int(np.min(profiles))
    profile_end = int(np.max(profiles))

    with CALIOPRegularGridReader(
        product="L1",
        version=version,
        data_type=data_type or CALIOP_L1_PRODUCT_TYPE,
        granule_date=granule_id,
        grid=grid,
        slice_start=profile_start,
        slice_end=profile_end,
        slice_start_end_type="profindex",
        index30m_alt_max=len(altitudes),
        folderpath=str(l1_path.parent),
    ) as reader:
        arrays = {}
        for name in SIGNAL_VARIABLES:
            array = reader.get_data(name).sel(profile=profiles)
            if array.sizes["altitude"] != len(altitudes):
                raise ValueError(
                    f"Incompatible vertical grid for {name}: "
                    f"{array.sizes['altitude']} versus {len(altitudes)}."
                )
            arrays[name] = array.assign_coords(altitude=altitudes).load()

    return xr.Dataset(arrays)

In [ ]:
# decode_times=False prevents xarray from attempting to decode the TAI calendar.
product_ds = xr.open_dataset(NETCDF_PATH, decode_times=False)
product_ds = normalize_product_dataset(product_ds)

profiles = product_ds.coords["profile"].values
altitudes = product_ds.coords["altitude"].values
latitudes = product_ds["Latitude"].values.squeeze()
longitudes = product_ds["Longitude"].values.squeeze()

l1_path = locate_l1_file(
    NETCDF_PATH,
    product_ds,
    COMMON_CONFIG_PATH,
    explicit_path=L1_PATH,
)
l1_ds_333m = load_l1_signals(
    l1_path, profiles, altitudes, grid="333mx30m"
)
l1_ds_5km = load_l1_signals(
    l1_path, profiles, altitudes, grid="5kmx60m"
)

print(f"2D-McDA product : {NETCDF_PATH}")
print(f"CALIOP L1 file  : {l1_path}")
print(f"Profiles        : {profiles[0]} to {profiles[-1]} ({len(profiles)})")
print(f"Altitudes       : {altitudes[0]:.2f} to {altitudes[-1]:.2f} km")

In [ ]:
MAX_DETECTION_LEVEL = 5
CHANNEL_FLAG_VALUES = (0, 1, 2, 3, 4, 5, 250, 251, 252, 253, 254)
CHANNEL_FLAG_LABELS = (
    ["No detection"]
    + [f"Detection level {level}" for level in range(1, MAX_DETECTION_LEVEL + 1)]
    + [
        "Low confidence small strips",
        "Almost fully attenuated",
        "Fully attenuated",
        "Likely artifact",
        "Surface",
    ]
)
channel_palette = sns.cubehelix_palette(
    MAX_DETECTION_LEVEL,
    start=2,
    rot=1,
    hue=1.0,
    gamma=1.0,
    light=0.8,
    dark=0.2,
    reverse=True,
)
channel_palette.insert(0, (1.0, 1.0, 1.0))
channel_palette.extend(
    [
        (0.7, 0.0, 0.0),
        (0.8, 0.0, 0.0),
        (1.0, 0.0, 0.0),
        (0.8, 0.8, 0.8),
        (0.5, 0.0, 0.0),
    ]
)
CHANNEL_FLAG_COLORS = [to_hex(color) for color in channel_palette]

COMPOSITE_FLAG_VALUES = (1, 2, 3, 5, 7)
COMPOSITE_FLAG_LABELS = (
    "Clear air",
    "Atmospheric feature",
    "Low confidence",
    "Surface/Subsurface",
    "Fully attenuated",
)
COMPOSITE_FLAG_COLORS = (
    "#ffffff",
    "#00539c",
    to_hex((0.6, 0.6, 0.6)),
    to_hex((88 / 255, 41 / 255, 0 / 255)),
    to_hex((222 / 255, 41 / 255, 22 / 255)),
)
BACKSCATTER_BOUNDS = [
    float(value) for value in cmlidar.cm.BACKSCATTER_DISCRETE_BOUNDS
]
BACKSCATTER_CMAP = cmlidar.cm.backscatter_18
BACKSCATTER_PALETTE = (
    [to_hex(BACKSCATTER_CMAP.get_under())]
    + [to_hex(BACKSCATTER_CMAP(index)) for index in range(BACKSCATTER_CMAP.N)]
    + [to_hex(BACKSCATTER_CMAP.get_over())]
)
BACKSCATTER_COLOR_COUNT = len(BACKSCATTER_PALETTE)


def format_latitude(value):
    return f"{abs(value):.2f}° {'S' if value < 0 else 'N'}"


def format_longitude(value):
    return f"{abs(value):.2f}° {'W' if value < 0 else 'E'}"


def latitude_longitude_ticks(count=6):
    indices = np.linspace(0, len(profiles) - 1, count, dtype=int)
    return [
        (
            float(profiles[index]),
            f"{format_latitude(latitudes[index])} / "
            f"{format_longitude(longitudes[index])}",
        )
        for index in indices
    ]


LAT_LON_TICKS = latitude_longitude_ticks()


def format_scientific(value):
    exponent = int(np.floor(np.log10(value)))
    mantissa = value / 10**exponent
    superscripts = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")
    return f"{mantissa:g}×10{str(exponent).translate(superscripts)}"


def colorbar_hook(labels, font_size="7pt"):
    def hook(plot, element):
        colorbar = plot.handles.get("colorbar")
        if colorbar is None:
            return
        colorbar.ticker = FixedTicker(ticks=list(labels))
        colorbar.major_label_overrides = labels
        colorbar.major_label_text_font_size = font_size
    return hook


def mask_plot(array, title, categories, category_labels, palette):
    values = array.transpose("altitude", "profile").values
    categories = np.asarray(categories)
    classes = np.full(values.shape, np.nan, dtype=float)
    for index, category in enumerate(categories):
        classes[values == category] = index

    labels = {
        float(index): label
        for index, label in enumerate(category_labels)
    }
    hover = HoverTool(
        tooltips=[
            ("Profile", "$x{0}"),
            ("Altitude", "$y{0.000} km"),
            ("Flag", "@flag{0}"),
        ]
    )
    image = hv.Image(
        (profiles, altitudes, classes, values),
        kdims=["profile", "altitude"],
        vdims=["color_class", "flag"],
    )
    return image.opts(
        hv.opts.Image(
            cmap=list(palette),
            clim=(-0.5, len(categories) - 0.5),
            colorbar=True,
            clabel="Detection class",
            xticks=LAT_LON_TICKS,
            xlabel="Latitude / Longitude",
            ylabel="Altitude (km)",
            title=title,
            width=PLOT_WIDTH,
            height=PLOT_HEIGHT,
            ylim=(
                ALTITUDE_RANGE
                if ALTITUDE_RANGE is not None
                else (float(np.min(altitudes)), float(np.max(altitudes)))
            ),
            tools=[hover],
            hooks=[colorbar_hook(labels, font_size="6pt")],
        )
    )


def composite_mask_plot(array, title):
    raw_values = array.transpose("altitude", "profile").values
    finite = np.isfinite(raw_values)
    base_values = np.full(raw_values.shape, np.nan, dtype=float)
    base_values[finite] = np.bitwise_and(
        raw_values[finite].astype(np.uint8), 7
    )
    classes = np.full(raw_values.shape, np.nan, dtype=float)
    for index, category in enumerate(COMPOSITE_FLAG_VALUES):
        classes[base_values == category] = index

    labels = {
        float(index): label
        for index, label in enumerate(COMPOSITE_FLAG_LABELS)
    }
    hover = HoverTool(
        tooltips=[
            ("Profile", "$x{0}"),
            ("Altitude", "$y{0.000} km"),
            ("Composite flag", "@composite_flag{0}"),
            ("Base class (bits 1–3)", "@base_class{0}"),
        ]
    )
    image = hv.Image(
        (profiles, altitudes, classes, base_values, raw_values),
        kdims=["profile", "altitude"],
        vdims=["color_class", "base_class", "composite_flag"],
    )
    return image.opts(
        hv.opts.Image(
            cmap=list(COMPOSITE_FLAG_COLORS),
            clim=(-0.5, len(COMPOSITE_FLAG_VALUES) - 0.5),
            colorbar=True,
            clabel="Composite class",
            xticks=LAT_LON_TICKS,
            xlabel="Latitude / Longitude",
            ylabel="Altitude (km)",
            title=title,
            width=PLOT_WIDTH,
            height=PLOT_HEIGHT,
            ylim=(
                ALTITUDE_RANGE
                if ALTITUDE_RANGE is not None
                else (float(np.min(altitudes)), float(np.max(altitudes)))
            ),
            tools=[hover],
            hooks=[colorbar_hook(labels, font_size="6pt")],
        )
    )


def backscatter_plot(array, title):
    values = array.transpose("altitude", "profile").values
    classes = np.full(values.shape, np.nan, dtype=float)
    valid = np.isfinite(values)
    classes[valid] = np.digitize(values[valid], bins=BACKSCATTER_BOUNDS)
    labels = {
        index + 0.5: format_scientific(bound)
        for index, bound in enumerate(BACKSCATTER_BOUNDS)
    }
    labels[0.0] = f"▼ < {format_scientific(BACKSCATTER_BOUNDS[0])}"
    labels[float(BACKSCATTER_COLOR_COUNT - 1)] = (
        f"▲ > {format_scientific(BACKSCATTER_BOUNDS[-1])}"
    )
    hover = HoverTool(
        tooltips=[
            ("Profile", "$x{0}"),
            ("Altitude", "$y{0.000} km"),
            ("β′", "@backscatter{0.000000e} km⁻¹ sr⁻¹"),
        ]
    )
    image = hv.Image(
        (profiles, altitudes, classes, values),
        kdims=["profile", "altitude"],
        vdims=["color_class", "backscatter"],
    )
    return image.opts(
        hv.opts.Image(
            cmap=BACKSCATTER_PALETTE,
            clim=(-0.5, BACKSCATTER_COLOR_COUNT - 0.5),
            colorbar=True,
            clabel="β′ (km⁻¹ sr⁻¹)",
            xticks=LAT_LON_TICKS,
            xlabel="Latitude / Longitude",
            ylabel="Altitude (km)",
            title=title,
            bgcolor="black",
            width=PLOT_WIDTH,
            height=PLOT_HEIGHT,
            ylim=(
                ALTITUDE_RANGE
                if ALTITUDE_RANGE is not None
                else (float(np.min(altitudes)), float(np.max(altitudes)))
            ),
            tools=[hover],
            hooks=[colorbar_hook(labels)],
        )
    )

In [ ]:
plots = [
    # Row 1: total 532 nm signal and composite mask.
    backscatter_plot(
        l1_ds_5km["Total_Attenuated_Backscatter_532"],
        "Total Attenuated Backscatter 532 nm — 5 km × 60 m",
    ),
    backscatter_plot(
        l1_ds_333m["Total_Attenuated_Backscatter_532"],
        "Total Attenuated Backscatter 532 nm — 333 m × 30 m",
    ),
    composite_mask_plot(
        product_ds["Composite_Detection_Flags"],
        "Composite Detection Flags — 333 m × 30 m",
    ),
    # Row 2: parallel 532 nm signal and mask.
    backscatter_plot(
        l1_ds_5km["Parallel_Attenuated_Backscatter_532"],
        "Parallel Attenuated Backscatter 532 nm — 5 km × 60 m",
    ),
    backscatter_plot(
        l1_ds_333m["Parallel_Attenuated_Backscatter_532"],
        "Parallel Attenuated Backscatter 532 nm — 333 m × 30 m",
    ),
    mask_plot(
        product_ds["Parallel_Detection_Flags_532"],
        "Parallel Detection Flags 532 nm — 333 m × 30 m",
        CHANNEL_FLAG_VALUES,
        CHANNEL_FLAG_LABELS,
        CHANNEL_FLAG_COLORS,
    ),
    # Row 3: perpendicular 532 nm signal and mask.
    backscatter_plot(
        l1_ds_5km["Perpendicular_Attenuated_Backscatter_532"],
        "Perpendicular Attenuated Backscatter 532 nm — 5 km × 60 m",
    ),
    backscatter_plot(
        l1_ds_333m["Perpendicular_Attenuated_Backscatter_532"],
        "Perpendicular Attenuated Backscatter 532 nm — 333 m × 30 m",
    ),
    mask_plot(
        product_ds["Perpendicular_Detection_Flags_532"],
        "Perpendicular Detection Flags 532 nm — 333 m × 30 m",
        CHANNEL_FLAG_VALUES,
        CHANNEL_FLAG_LABELS,
        CHANNEL_FLAG_COLORS,
    ),
    # Row 4: 1064 nm signal and mask.
    backscatter_plot(
        l1_ds_5km["Attenuated_Backscatter_1064"],
        "Attenuated Backscatter 1064 nm — 5 km × 60 m",
    ),
    backscatter_plot(
        l1_ds_333m["Attenuated_Backscatter_1064"],
        "Attenuated Backscatter 1064 nm — 333 m × 30 m",
    ),
    mask_plot(
        product_ds["Detection_Flags_1064"],
        "Detection Flags 1064 nm — 333 m × 30 m",
        CHANNEL_FLAG_VALUES,
        CHANNEL_FLAG_LABELS,
        CHANNEL_FLAG_COLORS,
    ),
]

## Display

All twelve panels share their `profile` and `altitude` ranges. Use the mouse wheel or Box Zoom tool in any panel; all other panels update automatically. The Reset button restores the initial extent.

In [ ]:
viewer = reduce(lambda left, right: left + right, plots).cols(3)
viewer = viewer.opts(
    hv.opts.Layout(
        shared_axes=True,
        merge_tools=True,
    )
)
viewer

In [ ]:
# Run this cell when the exploration is complete.
product_ds.close()